In [1]:
import os
import sys
import re

# =========================================================
# 1. ÉP VERSION PYSPARK (PHẢI CHẠY ĐẦU TIÊN TRƯỚC KHI IMPORT)
# =========================================================
modules_to_remove = [mod for mod in sys.modules if mod.startswith('pyspark') or mod.startswith('py4j')]
for mod in modules_to_remove: 
    del sys.modules[mod]

sys.path = [p for p in sys.path if "/usr/local/spark" not in p]
if "PYTHONPATH" in os.environ: 
    del os.environ["PYTHONPATH"]
    
conda_site_packages = "/opt/conda/lib/python3.13/site-packages"
if conda_site_packages not in sys.path: 
    sys.path.insert(0, conda_site_packages)
    
os.environ["SPARK_HOME"] = os.path.join(conda_site_packages, "pyspark")
os.environ["PYSPARK_PYTHON"] = "python3"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python3"


# =========================================================
# 2. BÂY GIỜ MỚI IMPORT PYSPARK VÀ KHỞI TẠO SESSION
# =========================================================
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, concat_ws, udf
from pyspark.sql.types import FloatType

spark = SparkSession.builder \
    .appName("VADER_tokenize_score_Pipeline") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.secret.key", "dataNLPmining-lab") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.my_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.my_catalog.type", "hadoop") \
    .config("spark.sql.catalog.my_catalog.warehouse", "s3a://raw-financial-data/iceberg_warehouse") \
    .getOrCreate()


# =========================================================
# 3. VÁ LỖI THỜI GIAN HADOOP
# =========================================================
hadoop_conf = spark._jsc.hadoopConfiguration()
iterator = hadoop_conf.iterator()
while iterator.hasNext():
    entry = iterator.next()
    val = str(entry.getValue()).strip().lower()
    match = re.fullmatch(r"(\d+)([smhd])", val)
    if match:
        num, unit = int(match.group(1)), match.group(2)
        ms_val = num * 1000 if unit == 's' else num * 60000 if unit == 'm' else num * 3600000 if unit == 'h' else num * 86400000
        hadoop_conf.set(entry.getKey(), str(ms_val))

print("✅ Khởi tạo Spark và môi trường hoàn tất!")

✅ Khởi tạo Spark và môi trường hoàn tất!


In [2]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [3]:
#file parquet nằm trong iceberg-warehouse
df_tokenize = spark.read.table("my_catalog.processed_zone_finnhub.news_tokens")
df_tokenize.show(5, truncate=False)

+---------+-------------------+---------------------------------------------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|id       |published_at       |title_tokens                                                                                       |summary_tokens                                                                                                                                            

In [4]:
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from pyspark.sql.functions import col, when, udf
from pyspark.sql.types import FloatType

# Tải bộ từ điển của VADER (Chỉ cần chạy 1 lần trên máy)
# nltk.download('vader_lexicon')

# Khởi tạo công cụ phân tích của VADER
sid = SentimentIntensityAnalyzer()

# 1. Tạo UDF lấy điểm tổng hợp (Compound) của VADER
# Điểm Compound kết hợp tất cả các yếu tố và chuẩn hóa về mức -1.0 đến 1.0
def get_vader_compound(text):
    if text is None or text == "":
        return 0.0
    # Gọi polarity_scores và chỉ trích xuất giá trị 'compound'
    scores = sid.polarity_scores(text)
    return scores['compound']

# Đăng ký UDF
udf_vader = udf(get_vader_compound, FloatType())


#processed
df_processed = df_tokenize \
    .withColumn("title_text", concat_ws(" ", col("title_tokens"))) \
    .withColumn("summary_text", concat_ws(" ", col("summary_tokens")))


# 2. Áp dụng UDF lên 2 cột text của bạn (title_text và summary_text)
# Giả sử df_processed là DataFrame đã được nối chuỗi ở bước trước
df_vader = df_processed \
    .withColumn("title_vader_score", udf_vader(col("title_text"))) \
    .withColumn("summary_vader_score", udf_vader(col("summary_text")))

# 3. Phân loại nhãn bằng VADER (Threshold khuyến nghị của chính tác giả VADER)
# Khác với TextBlob, VADER thường dùng mốc 0.05
VADER_POS_THRES = 0.05
VADER_NEG_THRES = -0.05

df_vader = df_vader.withColumn(
    "title_vader_sentiment",
    when(col("title_vader_score") >= VADER_POS_THRES, "Tích cực 🟢")
    .when(col("title_vader_score") <= VADER_NEG_THRES, "Tiêu cực 🔴")
    .otherwise("Trung tính ⚪")
)

df_vader = df_vader.withColumn(
    "summary_vader_sentiment",
    when(col("summary_vader_score") >= VADER_POS_THRES, "Tích cực 🟢")
    .when(col("summary_vader_score") <= VADER_NEG_THRES, "Tiêu cực 🔴")
    .otherwise("Trung tính ⚪")
)

# 4. Hiển thị kết quả
df_vader.select(
    "title_text", 
    "title_vader_score", "title_vader_sentiment",
    "summary_vader_score", "summary_vader_sentiment"
).show(40, truncate=40)

+----------------------------------------+-----------------+---------------------+-------------------+-----------------------+
|                              title_text|title_vader_score|title_vader_sentiment|summary_vader_score|summary_vader_sentiment|
+----------------------------------------+-----------------+---------------------+-------------------+-----------------------+
|market chatter jpmorgan google firms ...|              0.0|         Trung tính ⚪|                0.0|           Trung tính ⚪|
|uk regulator sets new rules google se...|           0.4019|          Tích cực 🟢|             0.9477|            Tích cực 🟢|
|google cloud model garden platform ex...|           0.4767|          Tích cực 🟢|             0.1027|            Tích cực 🟢|
|social media political problem time g...|          -0.4019|          Tiêu cực 🔴|            -0.6124|            Tiêu cực 🔴|
|custom ai chips coming nvidia crown c...|              0.0|         Trung tính ⚪|             0.1779|            Tíc

In [5]:

iceberg_table_path = "my_catalog.processed_zone_finnhub.news_VADER_tokens_sentiment_score"

# Đưa đúng tên biến iceberg_table_path vào saveAsTable
print(f"⏳ Đang ghi dữ liệu vào Apache Iceberg tại bảng: {iceberg_table_path} ...")
df_vader.write \
    .format("iceberg") \
    .mode("overwrite") \
    .saveAsTable(iceberg_table_path)

⏳ Đang ghi dữ liệu vào Apache Iceberg tại bảng: my_catalog.processed_zone_finnhub.news_VADER_tokens_sentiment_score ...
